# 08 - Visualization (Qualitative Results)

Reusable plotting functions for the paper's figures: beta-maps,
mixture-weight maps, intermediate feature maps (via forward hooks),
restoration comparison grids, and error heatmaps.

In [ ]:
import sys, os
sys.path.insert(0, "..")

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from utils.data import match_pairs, load_npy, downsample
from models.restoration_net import BaselineNet, DistributionMixtureRestorationNet
from pathlib import Path


device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
PROJECT_ROOT = Path("..").resolve()

TRAIN_GT_DIR = PROJECT_ROOT / "train" / "train" / "GT"
TRAIN_NOISY_DIR = PROJECT_ROOT / "train" / "train" / "NoisyLR"

RESULTS_DIR = PROJECT_ROOT / "results"
CKPT_DIR = RESULTS_DIR / "checkpoints"

pairs = match_pairs(TRAIN_GT_DIR, TRAIN_NOISY_DIR)
gt_path, noisy_path = pairs[0]
gt_img = load_npy(gt_path)
noisy_img = load_npy(noisy_path)
gt_ds = downsample(gt_img, noisy_img.shape[:2])

noisy_t = torch.from_numpy(noisy_img).float().unsqueeze(0).unsqueeze(0).to(device)
gt_t = torch.from_numpy(gt_img).float().unsqueeze(0).unsqueeze(0).to(device)


In [3]:
model = DistributionMixtureRestorationNet(use_film=True).to(device).eval()
ckpt = os.path.join(CKPT_DIR, "4_ldmh_plus_film.pt")
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device))
    print("Loaded trained checkpoint.")
else:
    print("No checkpoint found -- using randomly initialized weights (run Notebook 05/06 first for real figures).")

with torch.no_grad():
    restored, mix_weights, beta, scale = model(noisy_t)


Loaded trained checkpoint.


### Plot function: mixture-weight maps (one panel per component + dominant map)

In [4]:
def plot_mixture_weights(mix_weights, beta, save_path=None):
    K = mix_weights.shape[1]
    mw = mix_weights[0].detach().cpu().numpy()
    fig, axes = plt.subplots(1, K+1, figsize=(4*(K+1), 4))
    for k in range(K):
        im = axes[k].imshow(mw[k], cmap="viridis", vmin=0, vmax=1)
        axes[k].set_title(f"Component {k}\n(learned beta={beta[k].item():.3f})")
        axes[k].axis("off")
    dominant = mw.argmax(0)
    axes[K].imshow(dominant, cmap="tab10", vmin=0, vmax=K-1)
    axes[K].set_title("Dominant component per pixel")
    axes[K].axis("off")
    plt.colorbar(im, ax=axes[K-1], fraction=0.046)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

plot_mixture_weights(mix_weights, beta, save_path="../results/08_mixture_weights.png")


C:\Users\DELL\AppData\Local\Temp\ipykernel_35056\2433360298.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Plot function: intermediate feature maps via forward hooks

In [5]:
def plot_feature_maps(model, x, layer, n_channels=8, save_path=None):
    activations = {}
    def hook(module, inp, out):
        activations["feat"] = out.detach()
    handle = layer.register_forward_hook(hook)
    with torch.no_grad():
        model(x)
    handle.remove()

    feat = activations["feat"][0].cpu().numpy()  # [C, H, W]
    n_show = min(n_channels, feat.shape[0])
    fig, axes = plt.subplots(1, n_show, figsize=(2.2*n_show, 2.5))
    for i in range(n_show):
        axes[i].imshow(feat[i], cmap="viridis")
        axes[i].axis("off")
        axes[i].set_title(f"ch {i}", fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

# example: first LR-resolution NAFBlock inside the backbone
plot_feature_maps(model, noisy_t, model.backbone.lr_blocks[0], n_channels=8,
                   save_path="../results/08_feature_maps.png")


C:\Users\DELL\AppData\Local\Temp\ipykernel_35056\3563619965.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Plot function: restoration comparison grid across multiple models

In [6]:
@torch.no_grad()
def plot_restoration_grid(models_dict, noisy_t, gt_t, save_path=None):
    """models_dict: {name: model} -- each model(x) returns either pred or (pred, ...)."""
    n = len(models_dict) + 2
    fig, axes = plt.subplots(1, n, figsize=(3.2*n, 3.5))
    gt_np = gt_t[0,0].cpu().numpy()
    noisy_np = noisy_t[0,0].cpu().numpy()

    axes[0].imshow(gt_np, cmap="gray"); axes[0].set_title("GT"); axes[0].axis("off")
    axes[1].imshow(noisy_np, cmap="gray"); axes[1].set_title("Noisy input"); axes[1].axis("off")

    for i, (name, m) in enumerate(models_dict.items()):
        out = m(noisy_t)
        pred = out[0] if isinstance(out, tuple) else out
        pred_np = pred[0,0].cpu().numpy()
        from utils.metrics import psnr_np, ssim_np
        p = psnr_np(pred_np, gt_np)
        s = ssim_np(pred_np, gt_np)
        axes[i+2].imshow(pred_np, cmap="gray")
        axes[i+2].set_title(f"{name}\nPSNR={p:.2f} SSIM={s:.3f}")
        axes[i+2].axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

baseline_model = BaselineNet().to(device).eval()
bckpt = os.path.join(CKPT_DIR, "1_baseline.pt")
if os.path.exists(bckpt):
    baseline_model.load_state_dict(torch.load(bckpt, map_location=device))

plot_restoration_grid({"Baseline": baseline_model, "Full (LDMH+FiLM)": model},
                       noisy_t, gt_t, save_path="../results/08_restoration_grid.png")


C:\Users\DELL\AppData\Local\Temp\ipykernel_35056\463243773.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Plot function: per-pixel error heatmap

In [7]:
def plot_error_heatmap(pred_t, gt_t, save_path=None):
    err = (pred_t[0,0] - gt_t[0,0]).abs().cpu().numpy()
    plt.figure(figsize=(5,4.5))
    im = plt.imshow(err, cmap="inferno")
    plt.colorbar(im, fraction=0.046)
    plt.title("Absolute error heatmap")
    plt.axis("off")
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    return err

plot_error_heatmap(restored, gt_t, save_path="../results/08_error_heatmap.png")


C:\Users\DELL\AppData\Local\Temp\ipykernel_35056\3704184718.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


array([[0.06783158, 0.00536975, 0.01980376, ..., 0.00340623, 0.01036176,
        0.09797612],
       [0.04542622, 0.01662663, 0.01827919, ..., 0.00452584, 0.01392302,
        0.02856788],
       [0.00440937, 0.08136621, 0.10829771, ..., 0.04854527, 0.10826193,
        0.01760069],
       ...,
       [0.12098902, 0.08408108, 0.06476855, ..., 0.03738755, 0.00398418,
        0.07112709],
       [0.0850471 , 0.00340527, 0.00614536, ..., 0.04588795, 0.10953346,
        0.07480472],
       [0.01332387, 0.01425529, 0.02228802, ..., 0.04232657, 0.10531425,
        0.1399203 ]], shape=(256, 256), dtype=float32)